# Feature Engineering for Crop Yield Prediction

**Purpose:** Transform cleaned agricultural data into ML-ready features

**Outputs:**
- Feature-rich dataset with 15+ engineered features
- Publication-quality visualizations
- Actionable insights for agricultural policy

**Reference:** See `../../docs/05_FEATURE_ENGINEERING_GUIDE.md` for detailed documentation

## Setup

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Display options
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

print("✓ Imports complete")

In [ ]:
# Paths
current = Path.cwd()
project_root = current if (current / "CLAUDE.md").exists() else current.parent.parent

data_path = project_root / "data" / "processed"
output_path = project_root / "data" / "features"
figures_path = project_root / "notebooks" / "modeling" / "figures"

# Create output directories
output_path.mkdir(parents=True, exist_ok=True)
figures_path.mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root}")
print(f"Data path: {data_path}")
print(f"Output path: {output_path}")
print(f"Figures path: {figures_path}")

## Data Loading

In [ ]:
# Load processed datasets
print("Loading crop yield data...")
crop_yield_df1 = pd.read_pickle(data_path / "crop_yield1.pkl")
crop_yield_df2 = pd.read_pickle(data_path / "crop_yield2.pkl")
crop_yield_df = pd.concat([crop_yield_df1, crop_yield_df2], ignore_index=True)

print("Loading precipitation data...")
precipitation_df = pd.read_pickle(data_path / "precipitation.pkl")

print("Loading vegetation data...")
vegetation_df = pd.read_pickle(data_path / "vegetation.pkl")

print(f"\nDataset sizes:")
print(f"  Crop Yield: {len(crop_yield_df):,} rows")
print(f"  Precipitation: {len(precipitation_df):,} rows")
print(f"  Vegetation: {len(vegetation_df):,} rows")

In [ ]:
# Data cleaning
print("Cleaning data...")

# Remove NaN production values
crop_yield_df = crop_yield_df[crop_yield_df['production'].notna()].copy()
print(f"After removing NaN production: {len(crop_yield_df):,} rows")

# Extract integer year from datetime
crop_yield_df['crop_year'] = crop_yield_df['year'].dt.year
precipitation_df['year_int'] = precipitation_df['year']
vegetation_df['year_int'] = vegetation_df['year']

print("✓ Data cleaning complete")

In [ ]:
# Data quality checks
print("\nData Quality Checks:")
print("="*60)

# Check for duplicates
cy_key = ["state", "district", "crop_year", "crop_name", "season"]
cy_dups = crop_yield_df.duplicated(cy_key, keep=False).sum()
print(f"Crop yield duplicates: {cy_dups:,}")
assert cy_dups == 0, "Found duplicates in crop yield data!"

# Check coverage
print(f"\nCoverage:")
print(f"  States: {crop_yield_df['state'].nunique()}")
print(f"  Districts: {crop_yield_df['district'].nunique()}")
print(f"  Crops: {crop_yield_df['crop_name'].nunique()}")
print(f"  Years: {crop_yield_df['crop_year'].min()} - {crop_yield_df['crop_year'].max()}")

print("\n✓ Quality checks passed")

## Analysis 1: Lagged Yield Features (HIGHEST PRIORITY)

Generate historical yield predictors - the single most important feature set for ML models.

In [ ]:
# Sort data for proper lag calculation
crop_yield_df = crop_yield_df.sort_values(
    ['state', 'district', 'crop_name', 'season', 'crop_year']
)

# Create lagged features (1, 2, 3 years)
print("Creating lagged yield features...")
for lag in [1, 2, 3]:
    crop_yield_df[f'yield_lag{lag}'] = crop_yield_df.groupby(
        ['state', 'district', 'crop_name', 'season']
    )['yield'].shift(lag)
    
    missing = crop_yield_df[f'yield_lag{lag}'].isna().sum()
    print(f"  yield_lag{lag}: {missing:,} missing ({missing/len(crop_yield_df)*100:.1f}%)")

# Year-over-year change rate
crop_yield_df['yield_change_1yr'] = (
    (crop_yield_df['yield'] - crop_yield_df['yield_lag1']) / 
    crop_yield_df['yield_lag1']
)

# 3-year rolling mean
crop_yield_df['yield_rolling_mean_3yr'] = crop_yield_df.groupby(
    ['state', 'district', 'crop_name', 'season']
)['yield'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())

print("\n✓ Lagged features created")
print(f"\nFeatures added: yield_lag1, yield_lag2, yield_lag3, yield_change_1yr, yield_rolling_mean_3yr")

In [ ]:
# Autocorrelation analysis
from statsmodels.graphics.tsaplots import plot_acf

# Top 5 crops by area
top_crops = crop_yield_df.groupby('crop_name', observed=True)['area'].sum().nlargest(5).index

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, crop in enumerate(top_crops):
    crop_data = crop_yield_df[
        crop_yield_df['crop_name'] == crop
    ]['yield'].dropna()
    
    plot_acf(crop_data, lags=5, ax=axes[idx], title=f'{crop} Yield Autocorrelation')
    axes[idx].set_xlabel('Lag (years)')
    axes[idx].set_ylabel('Autocorrelation')

axes[-1].axis('off')  # Hide extra subplot

plt.suptitle('Yield Persistence by Crop Type\n(Higher correlation = more predictable)', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(figures_path / '01_yield_autocorrelation.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization saved: 01_yield_autocorrelation.png")

## Analysis 2: Regional Performance Benchmarking (Z-Scores)

Identify underperforming districts relative to state averages.

In [ ]:
# Calculate state-level statistics
print("Calculating z-scores...")

crop_yield_df['state_mean_yield'] = crop_yield_df.groupby(
    ['state', 'crop_name', 'season', 'crop_year'], 
    observed=True
)['yield'].transform('mean')

crop_yield_df['state_std_yield'] = crop_yield_df.groupby(
    ['state', 'crop_name', 'season', 'crop_year'], 
    observed=True
)['yield'].transform('std')

# Z-score
crop_yield_df['yield_zscore'] = (
    (crop_yield_df['yield'] - crop_yield_df['state_mean_yield']) / 
    crop_yield_df['state_std_yield']
)

# Clip extreme outliers
crop_yield_df['yield_zscore_clipped'] = crop_yield_df['yield_zscore'].clip(-3, 3)

# Check distribution
print(f"\nZ-score statistics:")
print(crop_yield_df['yield_zscore'].describe())

print("\n✓ Z-scores calculated")

In [ ]:
# Identify chronic underperformers
print("Identifying chronic underperformers...")

chronic_underperformers = crop_yield_df[
    crop_yield_df['yield_zscore'] < -1
].groupby(['state', 'district', 'crop_name'], observed=True).size().reset_index(name='underperform_years')

chronic_underperformers = chronic_underperformers[
    chronic_underperformers['underperform_years'] >= 3
]

print(f"Found {len(chronic_underperformers)} district-crop combinations with 3+ years of underperformance")
print(f"\nTop 10 chronic underperformers:")
print(chronic_underperformers.sort_values('underperform_years', ascending=False).head(10))

# Merge flag back
crop_yield_df = crop_yield_df.merge(
    chronic_underperformers[['state', 'district', 'crop_name']].assign(
        is_chronic_underperformer=1
    ),
    on=['state', 'district', 'crop_name'],
    how='left'
)
crop_yield_df['is_chronic_underperformer'] = crop_yield_df['is_chronic_underperformer'].fillna(0)

print("\n✓ Chronic underperformer flag added")

In [ ]:
# State performance heatmap
print("Creating state performance heatmap...")

# Last 5 years only
recent_years = crop_yield_df['crop_year'] >= crop_yield_df['crop_year'].max() - 5

state_performance = crop_yield_df[recent_years].groupby(
    ['state', 'crop_name'], 
    observed=True
).agg(
    underperform_pct=('yield_zscore', lambda x: (x < -1).mean() * 100)
).reset_index()

# Pivot for heatmap
heatmap_data = state_performance.pivot(
    index='state', columns='crop_name', values='underperform_pct'
)

# Top 10 crops only
top_10_crops = crop_yield_df.groupby('crop_name', observed=True)['area'].sum().nlargest(10).index
heatmap_data = heatmap_data[top_10_crops]

# Plot
plt.figure(figsize=(12, 10))
sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='RdYlGn_r', 
            vmin=0, vmax=50, cbar_kws={'label': '% Districts Underperforming'})
plt.title(f'State Performance Heatmap ({crop_yield_df["crop_year"].max() - 5}-{crop_yield_df["crop_year"].max()})\n% of Districts with Yield Z-Score < -1',
          fontsize=14, fontweight='bold')
plt.xlabel('Crop')
plt.ylabel('State')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(figures_path / '02_state_performance_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization saved: 02_state_performance_heatmap.png")

## Analysis 3: Seasonal Weather Aggregation

Create precipitation features aligned with crop growing seasons.

In [ ]:
# TO BE IMPLEMENTED
# See ../../docs/05_FEATURE_ENGINEERING_GUIDE.md for detailed implementation steps

print("⚠️  Analysis 3 not yet implemented")
print("See guide for implementation: ../../docs/05_FEATURE_ENGINEERING_GUIDE.md")

## Feature Summary

Summary of all features generated so far.

In [ ]:
# List all engineered features
engineered_features = [
    'yield_lag1', 'yield_lag2', 'yield_lag3',
    'yield_change_1yr', 'yield_rolling_mean_3yr',
    'yield_zscore', 'yield_zscore_clipped',
    'state_mean_yield', 'state_std_yield',
    'is_chronic_underperformer'
]

print("\nEngineered Features Summary:")
print("="*60)
for feat in engineered_features:
    if feat in crop_yield_df.columns:
        non_null = crop_yield_df[feat].notna().sum()
        coverage = non_null / len(crop_yield_df) * 100
        print(f"  ✓ {feat:30s} {coverage:5.1f}% coverage")
    else:
        print(f"  ✗ {feat:30s} NOT FOUND")

print(f"\nTotal features created: {len([f for f in engineered_features if f in crop_yield_df.columns])}")

## Export Feature Dataset

Save feature-rich dataset for ML modeling.

In [ ]:
# Save to pickle (intermediate format)
output_file = output_path / "crop_yield_ml_features_v1.pkl"
crop_yield_df.to_pickle(output_file)
print(f"✓ Features saved to: {output_file}")

# Also save to Parquet (production format)
parquet_file = output_path / "crop_yield_ml_features_v1.parquet"
crop_yield_df.to_parquet(parquet_file, index=False)
print(f"✓ Features saved to: {parquet_file}")

print(f"\nFinal dataset size: {len(crop_yield_df):,} rows × {len(crop_yield_df.columns)} columns")

## Next Steps

1. **Complete remaining analyses:** Implement Analyses 3-8 from the guide
2. **Feature validation:** Check for data leakage, distributions
3. **Documentation:** Create feature dictionary with descriptions
4. **Model training:** Use feature dataset in ML pipeline

See `../../docs/05_FEATURE_ENGINEERING_GUIDE.md` for detailed instructions.